<a href="https://colab.research.google.com/github/nqa112/times-series-forecasting/blob/main/lstm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## BASIC SETUP

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torch.optim as optim

In [ ]:
# Set random seed
SEED = 12
torch.manual_seed(SEED)
np.random.seed(SEED)
# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using {device}.")

In [ ]:
import matplotlib.pyplot as plt
from matplotlib import rcParams
from cycler import cycler

rcParams['figure.figsize'] = 12, 4

# Box
rcParams['axes.spines.top'] = False
rcParams['axes.spines.left'] = False
rcParams['axes.spines.right'] = False
rcParams['axes.prop_cycle'] = cycler(color=['navy','goldenrod'])

# Grid and axis thickness, color
rcParams['axes.linewidth'] = 1
rcParams['axes.edgecolor'] = '#5B5859'
rcParams['axes.ymargin'] = 0
rcParams['axes.grid'] = True
rcParams['axes.grid.axis'] = 'y'
rcParams['axes.axisbelow'] = True
rcParams['grid.color'] = 'grey'
rcParams['grid.linewidth'] = 0.5

# xticks, yticks
rcParams['ytick.major.width'] = 0
rcParams['ytick.major.size'] = 0
rcParams['ytick.color'] = '#393433'
rcParams['xtick.major.width'] = 1
rcParams['xtick.major.size'] = 3
rcParams['xtick.color'] = '#393433'

# Line thickness
rcParams['lines.linewidth'] = 1.5

# Saving quality
rcParams['savefig.bbox'] = 'tight'
rcParams['savefig.dpi'] = 500
rcParams['savefig.transparent'] = True

## LOAD DATA

In [ ]:
csv_path = Path("BVData.csv")
df = pd.read_csv(csv_path)

# Convert into a DateTime type
df["DateTime"] = pd.to_datetime(df["DateTime"], format="%Y-%m-%d %H:%M")
df.drop(["Date", "Time"], axis=1, inplace=True)
df.head()

In [ ]:
df.info()

## DATA PREPROCESSING

### Feature engineering

#### Date related features

In [ ]:
# Day in a month
df["Day_of_month"] = df.DateTime.apply(lambda x: x.day)
# Day in a week
df["Day_of_week"] = df.DateTime.apply(lambda x: x.dayofweek)
# 24-hour based
df["Hour"] = df.DateTime.apply(lambda x: x.hour)
# Week in a year
df["Week"] = df.DateTime.apply(lambda x: x.week)

# Set "DateTime" column as row index
df = df.set_index("DateTime")
df.head()

In [ ]:
# Move target to the last column
target_feature = "QV"
df.insert(len(df.columns)-1, target_feature, df.pop(target_feature))

### Hyperparameters

In [ ]:
batch_size = 32
past_len = 24
pred_len = 12

input_size = df.shape[1]
output_size = 1
hidden_size = 64
num_layers = 1

num_epochs = 100
learning_rate = 5e-3
es_patience = 15
lr_patience = 5

model_save_path = "checkpoint_lstm.pth"

## DATASET PIPELINE

### Feature scaling
Use natural log transformation (with one plus element-wise since log does not accept zero)

In [ ]:
data_scaled = np.log1p(df.values)

### Train-Val-Test split

In [ ]:
train_ratio, val_ratio = 0.7, 0.2
train_size = int(len(df) * train_ratio)
val_size = int(len(df) * val_ratio)

In [ ]:
train_data = data_scaled[: train_size]
val_data = data_scaled[train_size : train_size+val_size]
test_data = data_scaled[train_size+val_size :]

print("Training Shape:", train_data.shape)
print("Validation Shape:", val_data.shape)
print("Testing Shape:", test_data.shape)

### Create Custom Dataset

#### Generate data sequences

In [ ]:
class TimeSeriesDataset(Dataset):
    def __init__(self, data, past_len=past_len, pred_len=pred_len):
        # Create data sequences
        data_len = data.shape[0]
        X, Y = list(), list()

        for i in range(data_len):
            input_end = i + past_len
            output_end = input_end + pred_len
            # check if we are beyond the dataset
            if output_end > data_len:
                break
            else:
                X.append(data[i:input_end])
                Y.append(data[input_end:output_end, -1])

        # Shape (samples, seq_len, features)
        self.X = np.array(X)
        # Shape (samples, seq_len, ) : univariate
        self.Y = np.array(Y)

    def __len__(self):
        # return the size of the dataset
        return len(self.X)

    def __getitem__(self, idx):
        # return one sample from the dataset
        features = self.X[idx]
        target = self.Y[idx]
        return features, target

In [ ]:
train_dataset = TimeSeriesDataset(train_data)
val_dataset = TimeSeriesDataset(val_data)
test_dataset = TimeSeriesDataset(test_data)

#### Generate from PyTorch's `DataLoader` class

In [ ]:
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

In [ ]:
X, y = next(iter(train_loader))

print("Features shape:", X.size())
print("Target shape:", y.size())

## MODEL AND LEARNING ALGORITHM

### Model architecture

In [ ]:
class LSTMModel(nn.Module):

    def __init__(self, input_size, pred_len, hidden_size, num_layers):
        super(LSTMModel, self).__init__()
        self.input_size = input_size
        self.pred_len = pred_len
        self.hidden_size = hidden_size # neurons in each lstm layer
        self.num_layers = num_layers # recurrent layers in the lstm
        # LSTM
        self.lstm = nn.LSTM(
            input_size=self.input_size,
            hidden_size=self.hidden_size,
            num_layers=self.num_layers,
            batch_first=True
        )
        # Output layer
        self.fc = nn.Linear(self.hidden_size, self.pred_len)


    def forward(self, x):
        # Initialize hidden and cell states with zeros
        # shape (num_layers, batch_size, hidden_size)
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(device)
        c0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(device)
        # Forward propagation
        # out: tensor of shape (batch_size, seq_length, hidden_size)
        out, ht = self.lstm(x, (h0, c0))
        # Decode the hidden state of the last time step
        out = self.fc(out[:, -1, :])

        return out

In [ ]:
model = LSTMModel(input_size, pred_len, hidden_size, num_layers).to(device)
total_params = sum(p.numel() for p in model.parameters())
learn_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(model)
print(f"\nTotal parameters: {total_params}")
print(f"Learnable parameters: {learn_params}")

### Loss and Optimizer

In [ ]:
loss_func = nn.MSELoss()
opt = optim.Adam(model.parameters(), lr=learning_rate)

### Callbacks

In [ ]:
# Early Stopping
# Stop training if validation loss does not improve
class EarlyStopping:

    def __init__(self, patience, model_save_path, min_delta=0):
        self.patience = patience
        self.min_delta = min_delta
        self.model_save_path = model_save_path
        self.counter = 0
        self.min_validation_loss = np.inf
        self.best_epoch = 0
        self.early_stop = False


    def __call__(self, epoch, model, validation_loss):
        delta_loss = self.min_validation_loss - validation_loss
        # Check if val loss is smaller than min loss
        if delta_loss > self.min_delta:
            self.min_validation_loss = validation_loss
            self.counter = 0
            # Save best model
            self.best_epoch = epoch
            torch.save(model.state_dict(), self.model_save_path)
        else:
            self.counter += 1
            if self.counter >= self.patience:
                print(f"Early Stopping.")
                print(f"Save best model at epoch {self.best_epoch}")
                self.early_stop = True

In [ ]:
# ReduceLROnPlateau
# Reduce learning rate when validation loss stops improving
lr_scheduler = optim.lr_scheduler.ReduceLROnPlateau(opt, factor=0.3, patience=lr_patience, verbose=True)

## TRAINING

### Define training loop

In [ ]:
def train_model(data_loader, model, loss_function, optimizer):
    num_batches = len(data_loader)
    total_loss = 0
    model.train()

    for X, y in data_loader:
        X, y = X.float().to(device), y.float().to(device)

        # Forward pass
        output = model(X)
        loss = loss_function(output, y)

        # Backward and optimize
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    train_avg_loss = total_loss / num_batches

    return train_avg_loss


def val_model(data_loader, model, loss_function):
    num_batches = len(data_loader)
    total_loss = 0

    model.eval()
    with torch.no_grad():
        for X, y in data_loader:
            X, y = X.float().to(device), y.float().to(device)

            output = model(X)
            total_loss += loss_function(output, y).item()

    val_avg_loss = total_loss / num_batches

    return val_avg_loss

In [ ]:
# Log losses for plotting
all_losses = []

# Initialize Early Stopping object
early_stopper = EarlyStopping(patience=es_patience, model_save_path=model_save_path)
for epoch in range(num_epochs):
    train_loss = train_model(train_loader, model, loss_func, opt)
    val_loss = val_model(val_loader, model, loss_func)
    all_losses.append([train_loss, val_loss])

    # Display
    print(f"\nEpoch [{epoch}/{num_epochs-1}]\t\tTrain loss: {train_loss:.6f} - Val loss: {val_loss:.6f}")

    # EarlyStopping
    early_stopper(epoch, model, val_loss)
    if early_stopper.early_stop:
        break
    # Adjust learning rate
    lr_scheduler.step(val_loss)

### Plot history

In [ ]:
plt.title("LSTM Model", size=18, y=1.1)
plt.plot(all_losses, label=["Train loss", "Val loss"])
plt.xlabel("Epoch", fontsize=13)
plt.ylabel("MSE", fontsize=13)
plt.legend(loc="upper right", fontsize=10)
plt.show()

## EVALUATION

### Load best checkpoint

In [ ]:
model.load_state_dict(torch.load(model_save_path))

### Predict

In [ ]:
def predict(data_loader, model):
    pred, true = torch.tensor([]), torch.tensor([])
    model.eval()
    with torch.no_grad():
        for X, y in data_loader:
            y_pred = model(X.float().to(device))
            pred = torch.cat((pred, y_pred.to(device)), 0)
            true = torch.cat((true, y.to(device)), 0)

    return pred, true

In [ ]:
y_pred, y_test = predict(test_loader, model)

# Inverse the transformation
y_pred_inv = np.expm1(y_pred)
y_test_inv = np.expm1(y_test)

### Metrics


In [ ]:
# Hours ahead to predict
forecast_length = 12

In [ ]:
truth = y_test_inv[:, forecast_length-1].numpy()
forecast = y_pred_inv[:, forecast_length-1].numpy()

diff = np.subtract(truth, forecast)
# MAPE
mape = np.mean(np.abs(diff / truth))
# RMSE
mse = np.mean(np.square(diff))
rmse = np.sqrt(mse)
# NSE
num = np.sum(np.square(diff))
den = np.sum(np.square(np.subtract(truth, truth.mean())))
nse = 1 - (num / den)

print(f"{forecast_length}-hour forecast MAPE : {mape:.4f}")
print(f"{forecast_length}-hour forecast RMSE: {rmse:.4f}")
print(f"{forecast_length}-hour forecast NSE: {nse:.4f}")

### Visualize forecasts

In [ ]:
plt.title(f"{forecast_length}-Hour Ahead Forecasting\nMAPE: {mape:.3f} - RMSE: {rmse:.3f} - NSE: {nse:.3f}", size=16, y=1.1)
plt.plot(truth, label="Ground Truth", color="teal")
plt.plot(forecast, label="Prediction", color="darkred")
plt.xlabel("Observation")
plt.legend(fontsize=10)
plt.show()